<a href="https://colab.research.google.com/github/khanhtran94/crawTTV/blob/master/predicete_vietlot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# =============================================================
# 1. TẢI DỮ LIỆU THẬT POWER 6/55 TỪ GITHUB
# =============================================================
url = "https://raw.githubusercontent.com/vietvudanh/vietlott-data/main/data/power655.jsonl"

print("Đang tải dữ liệu Power 6/55 từ GitHub...")
df = pd.read_json(url, lines=True)

# Lấy 6 số chính trong mỗi kỳ quay (bỏ qua số đặc biệt thứ 7)
data = [row[:6] for row in df['result']]

num_draws = len(data)
total_numbers = 55  # Power 6/55 gồm các số từ 1 đến 55

print(f"\n=== ĐÃ TẢI THÀNH CÔNG {num_draws} KỲ QUAY ===")
print("5 kỳ gần nhất từ dữ liệu thật:")
for i, d in enumerate(data[-5:], start=num_draws - 4):
    print(f"Kỳ {i}: {sorted(d)}")

# Hàm hỗ trợ tạo 3 vé dựa trên phân phối xác suất
def sample_ticket_from_probs(probabilities, num_tickets=3, temperature=1.0):
    probs = np.clip(probabilities, 1e-8, None)
    scaled_probs = np.exp(np.log(probs) / temperature)
    norm_probs = scaled_probs / np.sum(scaled_probs)

    tickets = []
    for _ in range(num_tickets):
        chosen = np.random.choice(range(1, total_numbers + 1), size=6, replace=False, p=norm_probs)
        tickets.append(sorted(list(chosen)))
    return tickets


# =============================================================
# 2. STRATEGY 2: LSTM (PYTORCH)
# =============================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size=55, hidden_size=64):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

seq_len = 5
X_seq, y_seq = [], []
for i in range(seq_len, len(data)):              # Iterate up to the second to last element
    seq_in = [np.eye(total_numbers)[np.array(draw) - 1].sum(axis=0) for draw in data[i - seq_len:i]]
    seq_out = np.eye(total_numbers)[np.array(data[i]) - 1].sum(axis=0)
    X_seq.append(seq_in)
    y_seq.append(seq_out)

X_seq_t = torch.tensor(np.array(X_seq), dtype=torch.float32)
y_seq_t = torch.tensor(np.array(y_seq), dtype=torch.float32)

lstm = LSTMModel(input_size=total_numbers)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.008)

print("\nĐang huấn luyện LSTM...")
lstm.train()
for epoch in range(60):
    optimizer.zero_grad()
    outputs = lstm(X_seq_t)
    loss = criterion(outputs, y_seq_t)
    loss.backward()
    optimizer.step()

last_seq = [np.eye(total_numbers)[np.array(draw) - 1].sum(axis=0) for draw in data[-seq_len:]]
last_seq_t = torch.tensor(np.array([last_seq]), dtype=torch.float32)

lstm.eval()
with torch.no_grad():
    logits = lstm(last_seq_t)
    lstm_probs = torch.sigmoid(logits).numpy()[0]

print("\n" + "="*40)
print("  STRATEGY 2: LSTM - 3 TICKETS")
print("="*40)
for idx, temp in enumerate([0.5, 0.7, 0.8, 1.0, 1.3, 1.5, 2.0], start=1):
    ticket = sample_ticket_from_probs(lstm_probs, num_tickets=1, temperature=temp)[0]
    print(f"Ticket {idx} (Temp {temp:.1f}): {ticket}")

Đang tải dữ liệu Power 6/55 từ GitHub...

=== ĐÃ TẢI THÀNH CÔNG 1402 KỲ QUAY ===
5 kỳ gần nhất từ dữ liệu thật:
Kỳ 1398: [24, 27, 32, 36, 38, 47]
Kỳ 1399: [6, 11, 25, 27, 37, 45]
Kỳ 1400: [4, 7, 11, 18, 22, 25]
Kỳ 1401: [1, 3, 9, 11, 41, 46]
Kỳ 1402: [1, 13, 23, 25, 26, 28]

Đang huấn luyện LSTM...

  STRATEGY 2: LSTM - 3 TICKETS
Ticket 1 (Temp 0.5): [np.int64(11), np.int64(13), np.int64(16), np.int64(30), np.int64(53), np.int64(55)]
Ticket 2 (Temp 0.7): [np.int64(11), np.int64(16), np.int64(32), np.int64(40), np.int64(47), np.int64(49)]
Ticket 3 (Temp 0.8): [np.int64(23), np.int64(44), np.int64(47), np.int64(48), np.int64(49), np.int64(52)]
Ticket 4 (Temp 1.0): [np.int64(17), np.int64(22), np.int64(33), np.int64(38), np.int64(52), np.int64(54)]
Ticket 5 (Temp 1.3): [np.int64(1), np.int64(10), np.int64(18), np.int64(25), np.int64(28), np.int64(55)]
Ticket 6 (Temp 1.5): [np.int64(11), np.int64(18), np.int64(21), np.int64(28), np.int64(40), np.int64(42)]
Ticket 7 (Temp 2.0): [np.int64(14

In [7]:
def check_winning_ticket(winning_numbers, generated_tickets):
    winning_set = set(winning_numbers)
    print("\n" + "="*40)
    print(f"  KIỂM TRA VÉ TRÚNG THƯỞNG")
    print(f"  Dãy số trúng thưởng: {sorted(winning_numbers)}")
    print("="*40)

    for i, ticket_info in enumerate(generated_tickets):
        ticket_numbers = set(ticket_info['ticket'])
        common_numbers = winning_set.intersection(ticket_numbers)
        num_matches = len(common_numbers)

        print(f"Vé {ticket_info['id']} (Temp {ticket_info['temp']:.1f}): {ticket_info['ticket']}")
        print(f"  -> Số trùng khớp: {num_matches} ({sorted(list(common_numbers))})")

# Ví dụ sử dụng hàm check_winning_ticket
# Bạn có thể thay đổi dãy số trúng thưởng này thành dãy số thực tế.
winning_numbers_example = [14, 18, 21, 38, 48, 52] # Đây là kết quả kỳ quay 1402 trong dữ liệu

# Lưu trữ các vé đã tạo cùng với thông tin temperature
generated_tickets_with_temp = []
for idx, temp in enumerate([0.5, 0.7, 0.8, 1.0, 1.3, 1.5, 2.0], start=1):
    ticket = sample_ticket_from_probs(lstm_probs, num_tickets=1, temperature=temp)[0]
    generated_tickets_with_temp.append({
        'id': idx,
        'temp': temp,
        'ticket': ticket
    })

check_winning_ticket(winning_numbers_example, generated_tickets_with_temp)



  KIỂM TRA VÉ TRÚNG THƯỞNG
  Dãy số trúng thưởng: [14, 18, 21, 38, 48, 52]
Vé 1 (Temp 0.5): [np.int64(21), np.int64(25), np.int64(28), np.int64(37), np.int64(38), np.int64(50)]
  -> Số trùng khớp: 2 ([np.int64(21), np.int64(38)])
Vé 2 (Temp 0.7): [np.int64(3), np.int64(13), np.int64(29), np.int64(32), np.int64(43), np.int64(48)]
  -> Số trùng khớp: 1 ([np.int64(48)])
Vé 3 (Temp 0.8): [np.int64(16), np.int64(27), np.int64(42), np.int64(49), np.int64(53), np.int64(55)]
  -> Số trùng khớp: 0 ([])
Vé 4 (Temp 1.0): [np.int64(5), np.int64(18), np.int64(19), np.int64(31), np.int64(32), np.int64(54)]
  -> Số trùng khớp: 1 ([np.int64(18)])
Vé 5 (Temp 1.3): [np.int64(11), np.int64(16), np.int64(21), np.int64(23), np.int64(28), np.int64(47)]
  -> Số trùng khớp: 1 ([np.int64(21)])
Vé 6 (Temp 1.5): [np.int64(5), np.int64(12), np.int64(15), np.int64(21), np.int64(38), np.int64(52)]
  -> Số trùng khớp: 3 ([np.int64(21), np.int64(38), np.int64(52)])
Vé 7 (Temp 2.0): [np.int64(8), np.int64(17), np.int